# Instructor Streaming — Live Extraction for Responsive UIs

**Week 1 | Notebook 3 of 4**

**What you'll learn:**
- `Partial[Model]` — streaming partial objects
- Building a live-updating extraction UI (Gradio)
- `create_iterable` — streaming lists of entities
- Async streaming — processing multiple documents in parallel
- Streaming from Anthropic vs OpenAI (API differences)
- Latency benchmarks: streaming vs. non-streaming UX

**Runtime:** ~45 minutes

In [ ]:
# 💰 COST ESTIMATE
from src.cost_tracker import print_cost_warning

print_cost_warning("01_instructor/03_streaming.ipynb")

## 1. Setup

In [ ]:
import instructor
from pydantic import BaseModel

from src.config import USE_OLLAMA, get_instructor_client

client = get_instructor_client("openai")

## 2. Partial Streaming — Yields Incomplete Objects

In [ ]:
class Order(BaseModel):
    customer: str
    items: list[str]
    total: float
    status: str


order_text = """
Order from John Doe: 2x MacBook Pro, 1x AirPods Pro.
Total: $4,298. Status: confirmed.
"""

print("Streaming partial objects:\n")
for partial_order in client.chat.completions.create_partial(
    model="gpt-4o-mini" if not USE_OLLAMA else "ollama/llama3.1",
    response_model=Order,
    messages=[{"role": "user", "content": f"Extract order: {order_text}"}],
):
    # Each iteration gives a progressively more complete object
    print(
        f"Customer: {partial_order.customer or '...'} | Items: {len(partial_order.items or [])} | Total: {partial_order.total or '...'}"
    )

## 3. Streaming Lists — Extract Multiple Items Progressively

In [ ]:
class LineItem(BaseModel):
    product_name: str
    quantity: int
    unit_price: float


invoice_text = """
Invoice #1234
- MacBook Pro x2 @ $1,999 each
- AirPods Pro x1 @ $249
- USB-C Cable x3 @ $19 each
"""

print("Streaming individual line items:\n")
for item in client.chat.completions.create_iterable(
    model="gpt-4o-mini" if not USE_OLLAMA else "ollama/llama3.1",
    response_model=LineItem,
    messages=[{"role": "user", "content": f"Extract all line items: {invoice_text}"}],
):
    print(f"  Got: {item.product_name} x{item.quantity} @ ${item.unit_price}")

## 4. Live-Updating Extraction UI with Gradio

In [ ]:
import gradio as gr


def extract_with_streaming(text: str):
    """Stream extraction results to Gradio UI."""

    class ExtractedInfo(BaseModel):
        name: str | None = None
        company: str | None = None
        amount: float | None = None
        date: str | None = None

    result_text = ""
    for partial in client.chat.completions.create_partial(
        model="gpt-4o-mini" if not USE_OLLAMA else "ollama/llama3.1",
        response_model=ExtractedInfo,
        messages=[{"role": "user", "content": f"Extract info from: {text}"}],
    ):
        result_text = partial.model_dump_json(indent=2)
        yield result_text


# Create Gradio interface
demo = gr.Interface(
    fn=extract_with_streaming,
    inputs=gr.Textbox(label="Input Text", value="John from Acme Corp paid $500 on 2025-01-15"),
    outputs=gr.Textbox(label="Extracted Info"),
    title="Live Extraction with Instructor Streaming",
    description="Type text and watch the structured data fill in live!",
)

# Uncomment to launch:
# demo.launch()

print("✅ Gradio demo ready. Uncomment demo.launch() to start the UI.")

## 5. Async Streaming — Processing Multiple Documents

In [ ]:
import asyncio

from openai import AsyncOpenAI

async_client = instructor.from_openai(AsyncOpenAI())


class User(BaseModel):
    name: str
    age: int
    city: str


async def extract_user(text: str) -> User:
    return await async_client.chat.completions.create(
        model="gpt-4o-mini" if not USE_OLLAMA else "ollama/llama3.1",
        response_model=User,
        messages=[{"role": "user", "content": text}],
    )


texts = [
    "Alice, 30, lives in New York",
    "Bob is 25 and from San Francisco",
    "Carol, 35, based in Chicago",
]


async def process_batch():
    tasks = [extract_user(t) for t in texts]
    results = await asyncio.gather(*tasks)
    return results


results = asyncio.run(process_batch())
for user in results:
    print(f"{user.name}, {user.age}, {user.city}")

## 6. Streaming API Differences: OpenAI vs Anthropic

In [ ]:
# OpenAI streaming (shown above) uses create_partial()

# Anthropic streaming is identical — same API!
# from anthropic import Anthropic
# anthropic_client = instructor.from_anthropic(Anthropic())
# for partial in anthropic_client.chat.completions.create_partial(...):
#     ...

print("✅ Instructor abstracts streaming differences between providers")
print("   Same code works for OpenAI, Anthropic, and local models")

## 7. Latency Benchmark: Streaming vs Non-Streaming UX

In [ ]:
import time


class Summary(BaseModel):
    title: str
    key_points: list[str]


long_text = "Artificial intelligence is transforming..." * 50  # Long text

# Non-streaming: wait for everything
start = time.time()
summary = client.chat.completions.create(
    model="gpt-4o-mini" if not USE_OLLAMA else "ollama/llama3.1",
    response_model=Summary,
    messages=[{"role": "user", "content": f"Summarize: {long_text[:500]}"}],
)
non_stream_time = time.time() - start

# Streaming: first field appears immediately
start = time.time()
first_field_time = None
for partial in client.chat.completions.create_partial(
    model="gpt-4o-mini" if not USE_OLLAMA else "ollama/llama3.1",
    response_model=Summary,
    messages=[{"role": "user", "content": f"Summarize: {long_text[:500]}"}],
):
    if partial.title and first_field_time is None:
        first_field_time = time.time() - start
        break

print(f"Non-streaming total time: {non_stream_time:.2f}s")
print(f"Streaming — first field visible: {first_field_time:.2f}s")
print(
    f"\n💡 UX improvement: users see data {non_stream_time / first_field_time:.1f}x faster with streaming"
)

## 8. Exercise: Build a Streaming Invoice Extractor

Create a Gradio app that streams invoice extraction with partial updates for each field.

In [ ]:
# YOUR TURN: Build a streaming invoice extractor UI

# class Invoice(BaseModel):
#     invoice_number: Optional[str] = None
#     customer: Optional[str] = None
#     total: Optional[float] = None
#     items: List[str] = []

# def stream_invoice(text: str):
#     for partial in client.chat.completions.create_partial(...):
#         yield partial.model_dump_json()

# gr.Interface(fn=stream_invoice, inputs="text", outputs="text").launch()

---

**Next:** [04_local_pipeline.ipynb](04_local_pipeline.ipynb) — Fully local structured extraction with Ollama